# MPO Block Decomposition Experiment

Tests the supervisor's proposal: approximate a large N-qubit unitary as a product of independent
3-qubit blocks (MPO with bond dimension χ), compile each block with genQC, and recombine.

## Workflow
1. Construct target unitaries with controllable entanglement across a chosen bipartition
2. Run the SVD-based block decomposition (sweep χ = 1, 2, …)
3. Measure how much fidelity is lost at step 2 alone (decomposition error)
4. Compile each 3-qubit block with genQC
5. Recombine the compiled circuits and evaluate end-to-end fidelity
6. Plot fidelity vs entanglement entropy to understand when the approach works

## Key insight
The decomposition fidelity at a bipartition is `σ_max² / Σ σ_k²` (χ=1 case).
This is computable *before* running genQC and tells you upfront whether the approach is viable.

## 1. Setup

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import svd
from IPython.display import display
from tqdm.auto import tqdm

from notebooks.shared.bootstrap import setup_notebook_paths
PROJECT_ROOT = setup_notebook_paths()

from notebooks.shared.evaluation_artifacts import make_artifact_dir, save_dataframe, save_figure, save_json
from my_genQC.inference.sampling import generate_compilation_tensors, decode_tensors_to_backend
from my_genQC.inference.evaluation_helper import get_unitaries
from my_genQC.inference.eval_metrics import UnitaryFrobeniusNorm, UnitaryInfidelityNorm
from my_genQC.platform.simulation import CircuitBackendType, Simulator
from my_genQC.platform.tokenizer.circuits_tokenizer import CircuitTokenizer
from my_genQC.pipeline.diffusion_pipeline import DiffusionPipeline
from my_genQC.utils.misc_utils import infer_torch_device

## 2. Config — edit only this cell

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
MODEL_DIR   = "./artifacts/models/unitary-baseline-reproduction/paper_unitary"
HF_REPO     = None   # set to e.g. "Floki00/qc_unitary_3qubit" to use a remote model
GATE_POOL   = ["h", "cx", "z", "x", "ccx", "swap"]

# ── Sampling ─────────────────────────────────────────────────────────────────
GUIDANCE_SCALE      = 7.5
SAMPLE_STEPS        = 20
MAX_GATES           = 12
SAMPLES_PER_BLOCK   = 128   # genQC samples per 3-qubit block
AUTO_BATCH_SIZE     = 128

# ── Experiment ───────────────────────────────────────────────────────────────
# Total qubits must be a multiple of BLOCK_SIZE
TOTAL_QUBITS  = 6       # 6 = 2 blocks of 3; 9 = 3 blocks of 3
BLOCK_SIZE    = 3       # qubits per block (must match genQC training)
NUM_TARGETS   = 20      # how many random target unitaries to test
BOND_DIMS     = [1, 2, 4]   # bond dimensions χ to sweep (1 = supervisor's proposal)
SEED          = 42

# ── Target unitary construction ───────────────────────────────────────────────
# "product"  : U = U_A ⊗ U_B  (zero entanglement, χ=1 is exact)
# "shallow"  : few random 2-qubit gates across the block boundary
# "deep"     : many cross-boundary gates (high entanglement)
# "random"   : Haar-random unitary
TARGET_MODES  = ["product", "shallow", "deep", "random"]

# ── Output ───────────────────────────────────────────────────────────────────
ARTIFACT_DIR  = make_artifact_dir(PROJECT_ROOT, "mpo-block-decomposition", "initial_sweep")
EXACT_TOL     = 1e-6    # infidelity threshold for "exact" compilation

print(f"Blocks: {TOTAL_QUBITS // BLOCK_SIZE} × {BLOCK_SIZE} qubits")
print(f"Artifacts → {ARTIFACT_DIR}")

## 3. Utilities

### 3a. Target unitary construction

In [ ]:
def random_unitary(n_qubits: int, rng: np.random.Generator) -> np.ndarray:
    """Haar-random unitary via QR decomposition of a random complex Gaussian matrix."""
    dim = 2 ** n_qubits
    Z = (rng.standard_normal((dim, dim)) + 1j * rng.standard_normal((dim, dim))) / np.sqrt(2)
    Q, R = np.linalg.qr(Z)
    phases = np.diag(R) / np.abs(np.diag(R))
    return Q * phases[np.newaxis, :]


def make_target_unitary(mode: str, total_qubits: int, block_size: int,
                        rng: np.random.Generator) -> np.ndarray:
    """
    Constructs an (2^N × 2^N) target unitary with varying entanglement across the
    block boundary, controlled by `mode`.
    """
    n_blocks = total_qubits // block_size

    if mode == "product":
        blocks = [random_unitary(block_size, rng) for _ in range(n_blocks)]
        U = blocks[0]
        for B in blocks[1:]:
            U = np.kron(U, B)
        return U

    if mode == "random":
        return random_unitary(total_qubits, rng)

    U = make_target_unitary("product", total_qubits, block_size, rng)
    n_cross = {"shallow": 2, "deep": 8}[mode]
    for _ in range(n_cross):
        q_A = int(rng.integers(0, block_size))
        q_B = block_size + int(rng.integers(0, block_size))
        G2 = random_unitary(2, rng)
        G_full = _embed_2q_gate(G2, q_A, q_B, total_qubits)
        U = G_full @ U
    return U


def _embed_2q_gate(G2: np.ndarray, q0: int, q1: int, n_qubits: int) -> np.ndarray:
    """Embeds a 2-qubit gate G2 acting on (q0, q1) into the full n_qubit space."""
    dim = 2 ** n_qubits
    G_full = np.eye(dim, dtype=complex)
    for col_bits in range(dim):
        for row_bits in range(dim):
            b0_row = (row_bits >> (n_qubits - 1 - q0)) & 1
            b1_row = (row_bits >> (n_qubits - 1 - q1)) & 1
            b0_col = (col_bits >> (n_qubits - 1 - q0)) & 1
            b1_col = (col_bits >> (n_qubits - 1 - q1)) & 1
            mask = ~((1 << (n_qubits - 1 - q0)) | (1 << (n_qubits - 1 - q1)))
            if (row_bits & mask) != (col_bits & mask):
                G_full[row_bits, col_bits] = 0.0
            else:
                i2 = b0_row * 2 + b1_row
                j2 = b0_col * 2 + b1_col
                G_full[row_bits, col_bits] = G2[i2, j2]
    return G_full

### 3b. MPO decomposition

In [ ]:
def operator_svd_bipartition(U: np.ndarray, n_qubits_A: int, n_qubits_B: int):
    """
    Schmidt / operator-SVD decomposition of a bipartite unitary.

    Writes U ≈ Σ_k σ_k  A_k ⊗ B_k  where A_k ∈ M(2^nA), B_k ∈ M(2^nB).

    Returns
    -------
    sigmas : (r,)        singular values (descending)
    As     : (r, dA, dA) left operator factors
    Bs     : (r, dB, dB) right operator factors
    """
    dA = 2 ** n_qubits_A
    dB = 2 ** n_qubits_B
    assert U.shape == (dA * dB, dA * dB), f"Expected ({dA*dB},{dA*dB}), got {U.shape}"

    # Reshape U[row_A*dB+row_B, col_A*dB+col_B] → T[row_A, row_B, col_A, col_B]
    T = U.reshape(dA, dB, dA, dB)
    # Permute to (row_A, col_A, row_B, col_B), flatten to M[(i_A,j_A), (i_B,j_B)]
    M = T.transpose(0, 2, 1, 3).reshape(dA * dA, dB * dB)

    Ul, sigmas, Vr = svd(M, full_matrices=False)
    # scipy: M = Ul @ diag(sigmas) @ Vr
    # Reconstruction: M[m,n] = Σ_k σ_k u_k[m] * conj(v_k)[n]
    #   where u_k = Ul[:,k]  and  v_k = conj(Vr[k]).
    # So B_k[i_B,j_B] = conj(v_k)[n] = Vr[k,n] — no extra .conj() on Vr.

    As = Ul.T.reshape(-1, dA, dA)   # As[k] = u_k.reshape(dA, dA)
    Bs = Vr.reshape(-1, dB, dB)     # Bs[k] = Vr[k].reshape(dB, dB)  (no .conj())

    return sigmas, As, Bs


def polar_to_unitary(M: np.ndarray) -> np.ndarray:
    """Projects a square matrix to the nearest unitary via polar decomposition."""
    U, _, Vh = svd(M)
    return U @ Vh


def mpo_block_decompose(U: np.ndarray, block_size: int, chi: int):
    """
    Decomposes an N-qubit unitary into blocks of `block_size` qubits using a
    sequential left-to-right MPO SVD with bond dimension χ.

    For χ=1 each block is a standalone unitary (fully independent).
    For χ>1 blocks are coupled via weighted combinations of rank-1 factors.

    Returns
    -------
    blocks   : list of (chi_eff, d, d) arrays — operator factors per block
    sigmas   : list of (chi_eff,) singular value arrays at each bond
    approx_U : reconstructed approximate unitary at the requested χ
    """
    total_qubits = int(np.round(np.log2(U.shape[0])))
    n_blocks = total_qubits // block_size
    assert n_blocks * block_size == total_qubits, "total_qubits must be divisible by block_size"

    if n_blocks == 1:
        return [U[np.newaxis]], [np.array([1.0])], U

    blocks = []
    bond_sigmas = []
    remaining = U
    n_remaining = total_qubits

    for b in range(n_blocks - 1):
        n_right = n_remaining - block_size
        sigs, As, Bs = operator_svd_bipartition(remaining, block_size, n_right)

        chi_eff = min(chi, len(sigs))
        sigs = sigs[:chi_eff]
        As   = As[:chi_eff]
        Bs   = Bs[:chi_eff]

        blocks.append(As)
        bond_sigmas.append(sigs)

        # Contract sigmas into the right factor; left factor stays scale-free.
        remaining = sum(sigs[k] * Bs[k] for k in range(chi_eff))
        n_remaining = n_right

    blocks.append(remaining[np.newaxis])
    bond_sigmas.append(np.array([1.0]))

    approx_U = _reconstruct_from_blocks(blocks)
    return blocks, bond_sigmas, approx_U


def _reconstruct_from_blocks(blocks):
    """Reconstructs the full approximate unitary from MPO block factors."""
    result = blocks[0][0]
    for b in range(1, len(blocks)):
        result = np.kron(result, blocks[b][0])
    return result


def decomposition_infidelity(U_orig: np.ndarray, U_approx: np.ndarray) -> float:
    """1 - |Tr(U_approx† U_orig)|² / dim²  (process infidelity)"""
    dim = U_orig.shape[0]
    overlap = np.abs(np.trace(U_approx.conj().T @ U_orig)) / dim
    return float(1.0 - overlap ** 2)


def sanity_check_svd(n_qubits=3, tol=1e-10):
    """Verify: product unitary must reconstruct with near-zero infidelity."""
    rng = np.random.default_rng(999)
    UA = random_unitary(n_qubits, rng)
    UB = random_unitary(n_qubits, rng)
    U  = np.kron(UA, UB)
    _, _, U_approx = mpo_block_decompose(U, block_size=n_qubits, chi=1)
    inf = decomposition_infidelity(U, U_approx)
    status = "PASS" if inf < tol else "FAIL"
    print(f"SVD sanity check [{status}]: product-unitary infidelity = {inf:.2e}  (tol={tol:.0e})")
    return inf < tol

sanity_check_svd()

### 3c. Entanglement diagnostics

In [ ]:
def operator_entanglement_entropy(U: np.ndarray, n_qubits_A: int, n_qubits_B: int) -> float:
    """
    Operator entanglement entropy S = -Σ p_k log p_k  where  p_k = σ_k² / Σ σ_j².
    S=0 means U is a product unitary; S=log(min(dA²,dB²)) is maximally entangled.
    """
    sigs, _, _ = operator_svd_bipartition(U, n_qubits_A, n_qubits_B)
    probs = sigs ** 2 / (sigs ** 2).sum()
    probs = probs[probs > 1e-15]
    return float(-np.sum(probs * np.log(probs)))


def chi1_fidelity_bound(U: np.ndarray, n_qubits_A: int, n_qubits_B: int) -> float:
    """
    Theoretical upper bound on fidelity achievable with χ=1:
    F_max = (σ_max / ‖σ‖)²
    """
    sigs, _, _ = operator_svd_bipartition(U, n_qubits_A, n_qubits_B)
    return float((sigs[0] / np.linalg.norm(sigs)) ** 2)

### 3d. genQC block compilation

In [ ]:
def load_pipeline(model_dir: str, hf_repo: str | None, device):
    if hf_repo:
        pipeline = DiffusionPipeline.from_pretrained(repo_id=hf_repo, device=device)
    else:
        p = Path(model_dir).resolve()
        config_path = str(p if p.is_dir() else p.parent) + "/"
        pipeline = DiffusionPipeline.from_config_file(config_path=config_path, device=device)
    pipeline.scheduler.set_timesteps(SAMPLE_STEPS)
    return pipeline


def unitary_to_split_tensor(U: np.ndarray) -> torch.Tensor:
    """Converts a complex numpy unitary to the [2, N, N] real split format expected by genQC."""
    U_c = U.astype(np.complex64)
    return torch.from_numpy(np.stack([U_c.real, U_c.imag], axis=0))


def make_prompt(gate_pool: list) -> str:
    return f"Compile circuit: {gate_pool}"


def compile_block_unitary(
    pipeline, simulator, tokenizer,
    U_block: np.ndarray,
    gate_pool: list,
    system_size: int,
    n_qubits: int,
    samples: int,
    max_gates: int,
    guidance_scale: float,
    auto_batch_size: int,
    device,
) -> dict:
    """
    Runs genQC on a single 3-qubit block unitary. Returns a dict with:
    - best_circuit   : compiled circuit with lowest infidelity (or None)
    - best_infidelity: infidelity of best circuit
    - all_circuits   : all valid decoded circuits
    - exact_found    : whether infidelity < EXACT_TOL
    """
    U_target = polar_to_unitary(U_block)
    U_split  = unitary_to_split_tensor(U_target).unsqueeze(0).float().to(device)
    prompt   = make_prompt(gate_pool)

    tensors = generate_compilation_tensors(
        pipeline=pipeline,
        prompt=[prompt],
        U=U_split,
        samples=samples,
        system_size=system_size,
        num_of_qubits=n_qubits,
        max_gates=max_gates,
        g=guidance_scale,
        auto_batch_size=auto_batch_size,
        enable_params=False,
        no_bar=True,
    )
    circuits, _ = decode_tensors_to_backend(
        simulator=simulator, tokenizer=tokenizer, tensors=tensors,
        params=None, silent=True, n_jobs=1, filter_errs=False,
    )
    valid = [c for c in circuits if c is not None]

    best_circuit    = None
    best_infidelity = float("inf")
    for c in valid:
        U_c = simulator.backend.get_unitary(c)
        inf = float(UnitaryInfidelityNorm.distance(
            torch.as_tensor(U_c), torch.as_tensor(U_target)
        ).item())
        if inf < best_infidelity:
            best_infidelity = inf
            best_circuit    = c

    return {
        "best_circuit":    best_circuit,
        "best_infidelity": best_infidelity,
        "all_circuits":    valid,
        "exact_found":     best_infidelity < EXACT_TOL,
        "n_valid":         len(valid),
    }

### 3e. Circuit recombination

In [ ]:
def recombine_circuits(block_circuits: list, simulator) -> np.ndarray | None:
    """
    Combines per-block circuits into the full N-qubit unitary via tensor product.
    Returns None if any block failed to compile.
    """
    if any(c is None for c in block_circuits):
        return None
    unitaries = [simulator.backend.get_unitary(c) for c in block_circuits]
    result = unitaries[0]
    for U in unitaries[1:]:
        result = np.kron(result, U)
    return result

## 4. Load genQC model

In [ ]:
device = infer_torch_device()
print(f"Device: {device}")

pipeline  = load_pipeline(MODEL_DIR, HF_REPO, device)
simulator = Simulator(CircuitBackendType.QISKIT)

gate_pool  = GATE_POOL
vocabulary = {g: i for i, g in enumerate(gate_pool)}
tokenizer  = CircuitTokenizer(vocabulary)

system_size = pipeline.model.system_size if hasattr(pipeline.model, 'system_size') else BLOCK_SIZE
print(f"System size: {system_size}, block qubits: {BLOCK_SIZE}")

## 5. Decomposition diagnostics (no genQC yet)

First, check how much fidelity is lost at decomposition alone as a function of the target
unitary's entanglement. This runs fast and tells you whether it's worth running genQC.

In [ ]:
rng = np.random.default_rng(SEED)
n_qubits_A = BLOCK_SIZE
n_qubits_B = TOTAL_QUBITS - BLOCK_SIZE

diag_rows = []
for mode in TARGET_MODES:
    for i in range(NUM_TARGETS):
        U = make_target_unitary(mode, TOTAL_QUBITS, BLOCK_SIZE, rng)
        S  = operator_entanglement_entropy(U, n_qubits_A, n_qubits_B)
        F1 = chi1_fidelity_bound(U, n_qubits_A, n_qubits_B)
        _, _, U_approx = mpo_block_decompose(U, BLOCK_SIZE, chi=1)
        inf_decomp = decomposition_infidelity(U, U_approx)
        diag_rows.append({
            "mode": mode, "sample": i,
            "entanglement_entropy": S,
            "chi1_fidelity_bound": F1,
            "chi1_decomp_infidelity": inf_decomp,
        })

diag_df = pd.DataFrame(diag_rows)
display(diag_df.groupby("mode")[["entanglement_entropy", "chi1_fidelity_bound", "chi1_decomp_infidelity"]].mean().round(4))

In [ ]:
colors = {"product": "#5f8fcb", "shallow": "#63b17d", "deep": "#eb7d3c", "random": "#a06cc0"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=130)

ax = axes[0]
for mode, grp in diag_df.groupby("mode"):
    ax.scatter(grp["entanglement_entropy"], grp["chi1_fidelity_bound"],
               label=mode, alpha=0.75, s=40, color=colors[mode])
ax.set_xlabel("Operator entanglement entropy  S")
ax.set_ylabel("χ=1 fidelity upper bound")
ax.set_title("Fidelity bound vs entanglement")
ax.legend(); ax.grid(True, alpha=0.25)

ax = axes[1]
for mode, grp in diag_df.groupby("mode"):
    ax.scatter(grp["entanglement_entropy"], grp["chi1_decomp_infidelity"],
               label=mode, alpha=0.75, s=40, color=colors[mode])
ax.set_xlabel("Operator entanglement entropy  S")
ax.set_ylabel("χ=1 decomposition infidelity")
ax.set_title("Actual χ=1 decomposition error")
ax.legend(); ax.grid(True, alpha=0.25)

plt.tight_layout()
save_figure(fig, ARTIFACT_DIR / "decomposition_diagnostics.png")
plt.show()

## 6. Bond dimension sweep (decomposition error only)

Shows how decomposition infidelity improves as χ increases, without running genQC.

In [ ]:
rng2 = np.random.default_rng(SEED + 1)
bond_rows = []

for mode in TARGET_MODES:
    for i in range(min(NUM_TARGETS, 5)):
        U = make_target_unitary(mode, TOTAL_QUBITS, BLOCK_SIZE, rng2)
        S = operator_entanglement_entropy(U, n_qubits_A, n_qubits_B)
        for chi in BOND_DIMS:
            _, _, U_approx = mpo_block_decompose(U, BLOCK_SIZE, chi=chi)
            inf = decomposition_infidelity(U, U_approx)
            bond_rows.append({"mode": mode, "sample": i, "chi": chi,
                              "entanglement_entropy": S, "decomp_infidelity": inf})

bond_df = pd.DataFrame(bond_rows)
pivot = bond_df.groupby(["mode", "chi"])["decomp_infidelity"].mean().unstack("chi").round(4)
display(pivot)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), dpi=130)
for mode, grp in bond_df.groupby("mode"):
    means = grp.groupby("chi")["decomp_infidelity"].mean()
    ax.plot(means.index, means.values, marker="o", label=mode, color=colors[mode])
ax.set_xlabel("Bond dimension χ")
ax.set_ylabel("Mean decomposition infidelity")
ax.set_title("Decomposition error vs bond dimension")
ax.set_xticks(BOND_DIMS)
ax.legend(); ax.grid(True, alpha=0.25)
plt.tight_layout()
save_figure(fig, ARTIFACT_DIR / "bond_dim_sweep.png")
plt.show()

## 7. End-to-end: decompose → compile with genQC → recombine

Runs the full pipeline for χ=1. Skips targets where the decomposition infidelity
is already above `E2E_DECOMP_SKIP_THR` (no point running genQC on a bad approximation).

In [ ]:
E2E_CHI              = 1      # bond dim for end-to-end run
E2E_DECOMP_SKIP_THR  = 0.05  # skip if decomp infidelity > this
E2E_MODES            = ["product", "shallow"]  # modes to include
E2E_N_TARGETS        = 10    # targets per mode

In [ ]:
rng3 = np.random.default_rng(SEED + 2)
e2e_rows = []

for mode in E2E_MODES:
    for i in tqdm(range(E2E_N_TARGETS), desc=mode):
        U_target = make_target_unitary(mode, TOTAL_QUBITS, BLOCK_SIZE, rng3)
        S = operator_entanglement_entropy(U_target, n_qubits_A, n_qubits_B)

        blocks, sigmas, U_approx = mpo_block_decompose(U_target, BLOCK_SIZE, chi=E2E_CHI)
        decomp_inf = decomposition_infidelity(U_target, U_approx)

        row = {"mode": mode, "sample": i, "entanglement_entropy": S,
               "decomp_infidelity": decomp_inf, "skipped": False,
               "block_compile_infidelities": [], "e2e_infidelity": float("nan"),
               "all_blocks_exact": False}

        if decomp_inf > E2E_DECOMP_SKIP_THR:
            row["skipped"] = True
            e2e_rows.append(row)
            continue

        block_circuits = []
        block_infidelities = []
        for b_idx, block_factors in enumerate(blocks):
            U_block = polar_to_unitary(block_factors[0])
            result = compile_block_unitary(
                pipeline, simulator, tokenizer,
                U_block=U_block,
                gate_pool=gate_pool,
                system_size=system_size,
                n_qubits=BLOCK_SIZE,
                samples=SAMPLES_PER_BLOCK,
                max_gates=MAX_GATES,
                guidance_scale=GUIDANCE_SCALE,
                auto_batch_size=AUTO_BATCH_SIZE,
                device=device,
            )
            block_circuits.append(result["best_circuit"])
            block_infidelities.append(result["best_infidelity"])

        row["block_compile_infidelities"] = block_infidelities

        U_reconstructed = recombine_circuits(block_circuits, simulator)
        if U_reconstructed is not None:
            e2e_inf = decomposition_infidelity(U_target, U_reconstructed)
            row["e2e_infidelity"] = e2e_inf
            row["all_blocks_exact"] = all(inf < EXACT_TOL for inf in block_infidelities)

        e2e_rows.append(row)

e2e_df = pd.DataFrame(e2e_rows)
display(e2e_df[["mode", "entanglement_entropy", "decomp_infidelity",
                "e2e_infidelity", "all_blocks_exact", "skipped"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=130)

ax = axes[0]
for mode, grp in e2e_df[~e2e_df["skipped"]].groupby("mode"):
    ax.scatter(grp["decomp_infidelity"], grp["e2e_infidelity"],
               label=mode, alpha=0.8, s=50, color=colors[mode])
xmax = e2e_df["decomp_infidelity"].replace(float("inf"), np.nan).max() * 1.1 or 0.1
ax.plot([0, xmax], [0, xmax], "k--", alpha=0.4, label="x=y (decomp floor)")
ax.set_xlabel("Decomposition infidelity (floor)")
ax.set_ylabel("End-to-end infidelity")
ax.set_title("E2E vs decomposition floor")
ax.legend(); ax.grid(True, alpha=0.25)

ax = axes[1]
for mode, grp in e2e_df[~e2e_df["skipped"]].groupby("mode"):
    ax.scatter(grp["entanglement_entropy"], grp["e2e_infidelity"],
               label=mode, alpha=0.8, s=50, color=colors[mode])
ax.set_xlabel("Operator entanglement entropy  S")
ax.set_ylabel("End-to-end infidelity")
ax.set_title("E2E infidelity vs entanglement")
ax.legend(); ax.grid(True, alpha=0.25)

plt.tight_layout()
save_figure(fig, ARTIFACT_DIR / "e2e_results.png")
plt.show()

## 8. Summary table

In [ ]:
summary = e2e_df.groupby("mode").agg(
    n=("sample", "count"),
    n_skipped=("skipped", "sum"),
    mean_entropy=("entanglement_entropy", "mean"),
    mean_decomp_inf=("decomp_infidelity", "mean"),
    mean_e2e_inf=("e2e_infidelity", lambda x: x.dropna().mean()),
    frac_all_blocks_exact=("all_blocks_exact", "mean"),
).round(4)
display(summary)

save_dataframe(e2e_df.drop(columns=["block_compile_infidelities"]),
               ARTIFACT_DIR / "e2e_results.csv", index=False)
save_dataframe(summary.reset_index(), ARTIFACT_DIR / "summary.csv", index=False)
save_json({
    "total_qubits": TOTAL_QUBITS, "block_size": BLOCK_SIZE,
    "chi": E2E_CHI, "samples_per_block": SAMPLES_PER_BLOCK,
    "max_gates": MAX_GATES, "guidance_scale": GUIDANCE_SCALE,
    "exact_tol": EXACT_TOL, "seed": SEED,
}, ARTIFACT_DIR / "run_config.json")
print(f"Artifacts saved to {ARTIFACT_DIR}")